# Zaid Hybrid IDS — Step 10D: Full Teacher Validation
This self-contained notebook freezes Qwen/Qwen2.5-1.5B-Instruct and evaluates the corrected prompt on all 120 explainer-validation events. It embeds validation inputs, checks schema, detector-decision alignment, exact confidence, SHAP feature names and direction, unsupported features, prohibited content, and defensive action grounding. It does not access explainer-test outputs, save final targets, or modify the detector.

In [2]:
print('Embedded validation pilot: no Google Drive input file is required.')

Embedded validation pilot: no Google Drive input file is required.


In [3]:
!pip install -q transformers sentencepiece accelerate

In [4]:
from time import perf_counter
import base64
import json
import platform
import re
import sys
import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

TEACHER_MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
SEED = 20260829
VALIDATION_RECORDS_B64 = '...'
records = [
    json.loads(line)
    for line in base64.b64decode(VALIDATION_RECORDS_B64).decode('utf-8').splitlines()
    if line.strip()
]
ALL_FEATURES = ["Protocol","Flow Duration","Tot Fwd Pkts","Tot Bwd Pkts","TotLen Fwd Pkts","TotLen Bwd Pkts","Fwd Pkt Len Max","Fwd Pkt Len Min","Fwd Pkt Len Mean","Fwd Pkt Len Std","Bwd Pkt Len Max","Bwd Pkt Len Min","Bwd Pkt Len Mean","Bwd Pkt Len Std","Flow Byts/s","Flow Pkts/s","Flow IAT Mean","Flow IAT Std","Flow IAT Max","Flow IAT Min","Fwd IAT Tot","Fwd IAT Mean","Fwd IAT Std","Fwd IAT Max","Fwd IAT Min","Bwd IAT Tot","Bwd IAT Mean","Bwd IAT Std","Bwd IAT Max","Bwd IAT Min","Fwd Header Len","Bwd Header Len","Fwd Pkts/s","Bwd Pkts/s","Pkt Len Min","Pkt Len Max","Pkt Len Mean","Pkt Len Std","Pkt Len Var","FIN Flag Cnt","SYN Flag Cnt","RST Flag Cnt","PSH Flag Cnt","ACK Flag Cnt","URG Flag Cnt","ECE Flag Cnt","Down/Up Ratio","Pkt Size Avg","Init Fwd Win Byts","Init Bwd Win Byts","Fwd Act Data Pkts","Fwd Seg Size Min","Active Mean","Active Std","Active Max","Active Min","Idle Mean","Idle Std","Idle Max","Idle Min"]

assert len(records) == 120
assert all(record['split'] == 'validation' for record in records)
assert len(ALL_FEATURES) == 60
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Embedded validation events:', len(records))
print('Validation labels:', pd.Series([r['research_original_label'] for r in records]).value_counts().to_dict())
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: GPU is not enabled; stop before loading the model.')
print('Full validation setup: READY')

Embedded validation events: 120
Validation labels: {'FTP-BruteForce': 60, 'SSH-Bruteforce': 60}
Device: cuda
GPU: Tesla T4
Full validation setup: READY


In [5]:
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Loading teacher model:', TEACHER_MODEL_ID)
tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_ID)
teacher = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL_ID,
    dtype=torch.float16 if device.type == 'cuda' else torch.float32
).to(device)
teacher.eval()
teacher_parameter_count = int(sum(parameter.numel() for parameter in teacher.parameters()))
teacher_parameter_bytes = int(sum(
    parameter.numel() * parameter.element_size() for parameter in teacher.parameters()
))
print('Teacher parameters:', f'{teacher_parameter_count:,}')
print('In-memory parameter MB:', teacher_parameter_bytes / (1024 ** 2))
print('Teacher model: READY')

Loading teacher model: Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Teacher parameters: 1,543,714,304
In-memory parameter MB: 2944.4013671875
Teacher model: READY


In [6]:
BATCH_SIZE = 4 if device.type == 'cuda' else 1

SYSTEM_INSTRUCTION = '''You are a defensive intrusion-detection analyst. Follow the requested output schema exactly. Use only supplied detector and SHAP evidence. Do not infer a specific attack subtype, identity, endpoint, port, cause, or offensive procedure. SHAP is local model evidence and not causal proof. Return no preamble, bullets, markdown, or text after the required three lines.'''

def direction_requirement(first, second):
    first_supports = first['direction'] == 'supports_attack'
    second_supports = second['direction'] == 'supports_attack'
    if first_supports and second_supports:
        return (
            f"State that {first['name']} and {second['name']} provided the "
            "strongest local SHAP support for the detector decision."
        )
    if first_supports and not second_supports:
        return (
            f"State that {first['name']} supported the detector decision, while "
            f"{second['name']} opposed it."
        )
    if not first_supports and second_supports:
        return (
            f"State that {second['name']} supported the detector decision, while "
            f"{first['name']} opposed it."
        )
    return (
        f"State that {first['name']} and {second['name']} locally opposed the "
        "detector decision."
    )

def build_corrected_prompt(record):
    first = record['top_features'][0]
    second = record['top_features'][1]
    exact_confidence = f"{record['confidence']:.3f}"
    relation = direction_requirement(first, second)
    return f'''Write exactly three short lines.
Line 1 must begin Summary:. Line 2 must begin Evidence:. Line 3 must begin Action:.
Summary must contain the decision Attack and the exact confidence {exact_confidence}.
Evidence must copy these names exactly: {first['name']} and {second['name']}.
Do not mention any other feature name or any attack subtype.

Detector decision: Attack
Confidence: {exact_confidence}
Feature 1: {first['name']} (value={first['value']:.6g}, SHAP={first['shap_value']:+.6f}, direction={first['direction']})
Feature 2: {second['name']} (value={second['value']:.6g}, SHAP={second['shap_value']:+.6f}, direction={second['direction']})

Required evidence meaning: {relation}
Required action: advise reviewing network, authentication, and host logs and corroborating before containment.'''

prompts = []
for record in records:
    messages = [
        {'role': 'system', 'content': SYSTEM_INSTRUCTION},
        {'role': 'user', 'content': build_corrected_prompt(record)}
    ]
    prompts.append(tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    ))
tokenizer.padding_side = 'left'
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

teacher_raw_outputs = []
batch_times = []
print('Generating full validation explanations...')
generation_start = perf_counter()
for batch_start in range(0, len(prompts), BATCH_SIZE):
    batch_prompts = prompts[batch_start:batch_start + BATCH_SIZE]
    encoded = tokenizer(
        batch_prompts, return_tensors='pt', padding=True, truncation=True,
        max_length=768
    ).to(device)
    batch_start_time = perf_counter()
    with torch.inference_mode():
        generated = teacher.generate(
            **encoded, max_new_tokens=128, do_sample=False,
            repetition_penalty=1.05, pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    batch_times.append(perf_counter() - batch_start_time)
    continuation = generated[:, encoded['input_ids'].shape[1]:]
    teacher_raw_outputs.extend(
        tokenizer.batch_decode(continuation, skip_special_tokens=True)
    )
    completed = min(batch_start + BATCH_SIZE, len(prompts))
    if completed % 20 == 0 or completed == len(prompts):
        print(f'Completed {completed}/{len(prompts)}')
generation_seconds = perf_counter() - generation_start
assert len(teacher_raw_outputs) == len(records)
print('Total generation seconds:', generation_seconds)
print('Mean milliseconds per event:', generation_seconds / len(records) * 1000)
print('Full teacher validation generation: COMPLETE')

Generating full validation explanations...
Completed 20/120
Completed 40/120
Completed 60/120
Completed 80/120
Completed 100/120
Completed 120/120
Total generation seconds: 111.79807831700003
Mean milliseconds per event: 931.650652641667
Full teacher validation generation: COMPLETE


In [7]:
ip_pattern = re.compile(r'\b(?:\d{1,3}\.){3}\d{1,3}\b')
invented_port_pattern = re.compile(r'\bport\s+\d+\b', re.IGNORECASE)
unsupported_attack_pattern = re.compile(
    r'\b(?:FTP|SSH|brute[ -]?force|DDoS|malware|botnet|ransomware)\b',
    re.IGNORECASE
)
causal_pattern = re.compile(
    r'\b(?:proves?|caused|causes|definitely|confirmed attack|guarantees?)\b',
    re.IGNORECASE
)
unsafe_pattern = re.compile(
    r'\b(?:exploit|payload|reverse shell|credential dump|bypass authentication)\b',
    re.IGNORECASE
)

def normalize_schema(text):
    text = ' '.join(str(text).strip().split())
    text = re.sub(r'(?i)\bsummary\s*:', 'Summary:', text, count=1)
    text = re.sub(r'(?i)\s*\bevidence\s*:', '\nEvidence:', text, count=1)
    text = re.sub(r'(?i)\s*\baction\s*:', '\nAction:', text, count=1)
    return text.strip()

def validate_output(record, raw_output):
    normalized = normalize_schema(raw_output)
    lines = [line.strip() for line in normalized.splitlines() if line.strip()]
    schema_compliant = (
        len(lines) == 3 and
        lines[0].startswith('Summary:') and
        lines[1].startswith('Evidence:') and
        lines[2].startswith('Action:')
    )
    summary_line = lines[0] if len(lines) >= 1 else ''
    evidence_line = lines[1] if len(lines) >= 2 else ''
    action_line = lines[2] if len(lines) >= 3 else ''
    exact_confidence = f"{record['confidence']:.3f}"
    decision_aligned = 'attack' in summary_line.lower()
    confidence_aligned = exact_confidence in summary_line

    allowed_features = [feature['name'] for feature in record['top_features']]
    required_features = allowed_features[:2]
    evidence_lower = evidence_line.lower()
    mentioned_allowed = [
        feature for feature in allowed_features if feature.lower() in evidence_lower
    ]
    required_features_present = all(
        feature.lower() in evidence_lower for feature in required_features
    )
    mentioned_all_dataset_features = [
        feature for feature in ALL_FEATURES if feature.lower() in evidence_lower
    ]
    unsupported_features = sorted(
        set(mentioned_all_dataset_features) - set(allowed_features)
    )

    first, second = record['top_features'][:2]
    has_support_term = 'support' in evidence_lower
    has_opposition_term = any(term in evidence_lower for term in ['oppos', 'counter', 'reduc'])
    requires_support = any(
        feature['direction'] == 'supports_attack' for feature in [first, second]
    )
    requires_opposition = any(
        feature['direction'] == 'opposes_attack' for feature in [first, second]
    )
    direction_aligned = (
        (not requires_support or has_support_term) and
        (not requires_opposition or has_opposition_term)
    )
    action_grounded = (
        'review' in action_line.lower() and
        'corroborat' in action_line.lower()
    )

    prohibited_matches = []
    for category, pattern in [
        ('invented_ip', ip_pattern), ('invented_port', invented_port_pattern),
        ('unsupported_attack_type', unsupported_attack_pattern),
        ('causal_claim', causal_pattern), ('unsafe_content', unsafe_pattern)
    ]:
        if pattern.search(normalized):
            prohibited_matches.append(category)

    valid = bool(
        schema_compliant and decision_aligned and confidence_aligned and
        required_features_present and direction_aligned and action_grounded and
        len(unsupported_features) == 0 and len(prohibited_matches) == 0
    )
    return {
        'teacher_normalized_output': normalized,
        'schema_compliant': schema_compliant,
        'decision_aligned': decision_aligned,
        'confidence_aligned': confidence_aligned,
        'required_features_present': required_features_present,
        'allowed_feature_mentions': mentioned_allowed,
        'allowed_feature_mention_count': len(mentioned_allowed),
        'allowed_feature_coverage': len(mentioned_allowed) / len(allowed_features),
        'direction_aligned': direction_aligned,
        'action_grounded': action_grounded,
        'unsupported_feature_mentions': unsupported_features,
        'prohibited_matches': prohibited_matches,
        'teacher_valid': valid
    }

validated_records = []
for record, raw_output in zip(records, teacher_raw_outputs):
    validation = validate_output(record, raw_output)
    output_record = dict(record)
    output_record['teacher_model_id'] = TEACHER_MODEL_ID
    output_record['teacher_raw_output'] = raw_output
    output_record.update(validation)
    output_record['target_source'] = 'teacher' if validation['teacher_valid'] else 'grounded_fallback'
    output_record['target_text'] = (
        validation['teacher_normalized_output']
        if validation['teacher_valid'] else record['fallback_reference']
    )
    validated_records.append(output_record)

validation_df = pd.DataFrame([{
    'event_id': record['event_id'],
    'research_original_label': record['research_original_label'],
    'teacher_valid': record['teacher_valid'],
    'schema_compliant': record['schema_compliant'],
    'decision_aligned': record['decision_aligned'],
    'confidence_aligned': record['confidence_aligned'],
    'required_features_present': record['required_features_present'],
    'direction_aligned': record['direction_aligned'],
    'action_grounded': record['action_grounded'],
    'allowed_feature_coverage': record['allowed_feature_coverage'],
    'unsupported_feature_count': len(record['unsupported_feature_mentions']),
    'prohibited_match_count': len(record['prohibited_matches']),
    'target_source': record['target_source']
} for record in validated_records])

for metric in [
    'schema_compliant', 'decision_aligned', 'confidence_aligned',
    'required_features_present', 'direction_aligned', 'action_grounded',
    'teacher_valid'
]:
    print(f'{metric} rate:', float(validation_df[metric].mean()))
print('Fallback rate:', float(validation_df['target_source'].eq('grounded_fallback').mean()))
print('Mean allowed-feature coverage:', float(validation_df['allowed_feature_coverage'].mean()))
print('Unsupported-feature event rate:', float(validation_df['unsupported_feature_count'].gt(0).mean()))
print('Prohibited-content event rate:', float(validation_df['prohibited_match_count'].gt(0).mean()))
print('\nValidity by research label:')
print(pd.crosstab(
    validation_df['research_original_label'],
    validation_df['teacher_valid'],
    margins=True
))
print('Full teacher validation: COMPLETE')

schema_compliant rate: 1.0
decision_aligned rate: 1.0
confidence_aligned rate: 0.9583333333333334
required_features_present rate: 1.0
direction_aligned rate: 1.0
action_grounded rate: 1.0
teacher_valid rate: 0.9583333333333334
Fallback rate: 0.041666666666666664
Mean allowed-feature coverage: 0.4000000000000001
Unsupported-feature event rate: 0.0
Prohibited-content event rate: 0.0

Validity by research label:
teacher_valid            False  True  All
research_original_label                  
FTP-BruteForce               5    55   60
SSH-Bruteforce               0    60   60
All                          5   115  120
Full teacher validation: COMPLETE


In [8]:
failed_records = [
    record for record in validated_records
    if not record['teacher_valid']
]
print('Failed teacher records:', len(failed_records))
assert all(record['split'] == 'validation' for record in failed_records)

check_names = [
    'schema_compliant', 'decision_aligned', 'confidence_aligned',
    'required_features_present', 'direction_aligned', 'action_grounded'
]
for record in failed_records:
    failed_checks = [name for name in check_names if not record[name]]
    print('=' * 80)
    print('Research label used for QA selection only:', record['research_original_label'])
    print('Event:', record['event_id'])
    print('Expected confidence:', f"{record['confidence']:.3f}")
    print('Failed checks:', ', '.join(failed_checks) if failed_checks else 'none')
    print('RAW TEACHER OUTPUT:')
    print(record['teacher_raw_output'])
    print('NORMALIZED TEACHER OUTPUT:')
    print(record['teacher_normalized_output'])
    print('FINAL TARGET:')
    print(record['target_text'])
print('Failed validation diagnostic: COMPLETE')

Failed teacher records: 5
Research label used for QA selection only: FTP-BruteForce
Event: test-004081
Expected confidence: 0.998
Failed checks: confidence_aligned
RAW TEACHER OUTPUT:
Summary: Attack with high confidence
Evidence: Fwd Seg Size Min value=40, SHAP=+0.133168, supports_attack; Init Fwd Win Byts value=26883, SHAP=+0.089101, supports_attack
Action: Review network, authentication, and host logs, corroborate before containment.
NORMALIZED TEACHER OUTPUT:
Summary: Attack with high confidence
Evidence: Fwd Seg Size Min value=40, SHAP=+0.133168, supports_attack; Init Fwd Win Byts value=26883, SHAP=+0.089101, supports_attack
Action: Review network, authentication, and host logs, corroborate before containment.
FINAL TARGET:
Summary: The detector classified this flow as an attack with 0.998 confidence.
Evidence: Fwd Seg Size Min and Init Fwd Win Byts provided the strongest local support for the decision; the remaining listed SHAP features provide additional model evidence.
Action: 

In [9]:
example_indices = []
for label in ['FTP-BruteForce', 'SSH-Bruteforce']:
    label_indices = [
        index for index, record in enumerate(validated_records)
        if record['research_original_label'] == label
    ]
    example_indices.extend(label_indices[:2])

for index in example_indices:
    record = validated_records[index]
    print('=' * 80)
    print('Research label used for QA selection only:', record['research_original_label'])
    print('Event:', record['event_id'])
    print('Valid:', record['teacher_valid'], '| Target source:', record['target_source'])
    print('RAW TEACHER OUTPUT:')
    print(record['teacher_raw_output'])
    print('FINAL TARGET:')
    print(record['target_text'])
print('Stratified validation examples: REVIEWED')

Research label used for QA selection only: FTP-BruteForce
Event: test-002432
Valid: True | Target source: teacher
RAW TEACHER OUTPUT:
Summary: Attack with 1.000 confidence
Evidence: Fwd Seg Size Min value=40, SHAP=+0.126573, supports_attack; Init Fwd Win Byts value=26883, SHAP=+0.077877, supports_attack
Action: Review network, authentication, and host logs, corroborate before containment.
FINAL TARGET:
Summary: Attack with 1.000 confidence
Evidence: Fwd Seg Size Min value=40, SHAP=+0.126573, supports_attack; Init Fwd Win Byts value=26883, SHAP=+0.077877, supports_attack
Action: Review network, authentication, and host logs, corroborate before containment.
Research label used for QA selection only: FTP-BruteForce
Event: test-003434
Valid: True | Target source: teacher
RAW TEACHER OUTPUT:
Summary: Attack with 1.000 confidence
Evidence: Fwd Seg Size Min value=40, SHAP=+0.126572, supports_attack
Init Fwd Win Byts value=26883, SHAP=+0.077915, supports_attack
Action: Review network, authenti

In [10]:
assert set(record['split'] for record in validated_records) == {'validation'}
print('Validation only: no final target files were saved and explainer test was not accessed.')
print('If full validation passes, this exact model, prompt, generation configuration, and validator will be frozen for the final run.')

Validation only: no final target files were saved and explainer test was not accessed.
If full validation passes, this exact model, prompt, generation configuration, and validator will be frozen for the final run.
